In [3]:
"""
Hidden Markov Model POS Tagger with Brute Force Decoding
Ուսումնական նպատակների համար իրականացված HMM
"""

from collections import defaultdict
from itertools import product
import math


class HMMPOSTagger:
    """
    HMM-based Part-of-Speech Tagger with Brute Force Decoding
    """

    def __init__(self, smoothing=1.0):
        """
        Initialization

        Args:
            smoothing: Laplace smoothing parameter (default=1.0 for add-one smoothing)
        """
        self.smoothing = smoothing

        # Հավանականությունների պահպանման համար
        self.initial_probs = defaultdict(float)      # P(tag)
        self.transition_probs = defaultdict(float)   # P(tag_i | tag_{i-1})
        self.emission_probs = defaultdict(float)     # P(word | tag)

        # Քանակների պահպանում
        self.initial_counts = defaultdict(int)
        self.transition_counts = defaultdict(int)
        self.emission_counts = defaultdict(int)
        self.tag_counts = defaultdict(int)

        # Բոլոր tag-երի և բառերի ցուցակները
        self.all_tags = set()
        self.all_words = set()

        self.trained = False


    def train(self, training_data):
        """
        Train the HMM on tagged sentences

        Args:
            training_data: List of sentences, each sentence is a list of (word, tag) tuples
                          Example: [[("The", "DET"), ("dog", "NOUN")], ...]

        Հաշվարկում է:
        1. Initial probabilities - P(tag)
        2. Transition probabilities - P(tag_i | tag_{i-1})
        3. Emission probabilities - P(word | tag)
        """

        print("Training HMM POS Tagger...")
        print(f"Training on {len(training_data)} sentences")

        # ՔԱՅԼ 1: Հաշվել բոլոր դեպքերը
        for sentence in training_data:
            prev_tag = None

            for i, (word, tag) in enumerate(sentence):
                # Ավելացնել tag-ը և բառը մեր ցուցակներին
                self.all_tags.add(tag)
                self.all_words.add(word)

                # Հաշվել initial counts (առաջին բառի tag-ը)
                if i == 0:
                    self.initial_counts[tag] += 1

                # Հաշվել transition counts
                if prev_tag is not None:
                    self.transition_counts[(prev_tag, tag)] += 1

                # Հաշվել emission counts
                self.emission_counts[(tag, word)] += 1

                # Հաշվել tag counts
                self.tag_counts[tag] += 1

                prev_tag = tag

        # ՔԱՅԼ 2: Հաշվել հավանականությունները smoothing-ով

        num_tags = len(self.all_tags)
        num_words = len(self.all_words)
        total_sentences = len(training_data)

        # Initial probabilities: P(tag)
        for tag in self.all_tags:
            count = self.initial_counts[tag]
            # Add-one smoothing
            self.initial_probs[tag] = (count + self.smoothing) / (total_sentences + self.smoothing * num_tags)

        # Transition probabilities: P(tag_i | tag_{i-1})
        for prev_tag in self.all_tags:
            for curr_tag in self.all_tags:
                count = self.transition_counts[(prev_tag, curr_tag)]
                prev_count = self.tag_counts[prev_tag]
                # Add-one smoothing
                self.transition_probs[(prev_tag, curr_tag)] = (
                    (count + self.smoothing) / (prev_count + self.smoothing * num_tags)
                )

        # Emission probabilities: P(word | tag)
        for tag in self.all_tags:
            for word in self.all_words:
                count = self.emission_counts[(tag, word)]
                tag_count = self.tag_counts[tag]
                # Add-one smoothing
                self.emission_probs[(tag, word)] = (
                    (count + self.smoothing) / (tag_count + self.smoothing * num_words)
                )

        self.trained = True

        print(f"Training complete!")
        print(f"Learned {num_tags} POS tags: {sorted(self.all_tags)}")
        print(f"Vocabulary size: {num_words} words")
        print()


    def calculate_sequence_probability(self, words, tags):
        """
        Calculate the probability of a tag sequence for given words

        P(tags, words) = P(tag_1) × P(word_1|tag_1) × P(tag_2|tag_1) × P(word_2|tag_2) × ...

        Args:
            words: List of words
            tags: List of tags (same length as words)

        Returns:
            Probability of the sequence (float)
        """

        if len(words) != len(tags):
            raise ValueError("Words and tags must have the same length")

        # Օգտագործում ենք log probabilities՝ underflow-ից խուսափելու համար
        log_prob = 0.0

        for i, (word, tag) in enumerate(zip(words, tags)):
            if i == 0:
                # Initial probability
                prob = self.initial_probs.get(tag, self.smoothing / (len(self.all_tags) * self.smoothing))
            else:
                # Transition probability
                prev_tag = tags[i-1]
                prob = self.transition_probs.get((prev_tag, tag),
                                                  self.smoothing / (len(self.all_tags) * self.smoothing))

            # Emission probability
            emission_prob = self.emission_probs.get((tag, word),
                                                     self.smoothing / (len(self.all_words) * self.smoothing))

            # Log space-ում՝ բազմապատկումը դառնում է գումարում
            log_prob += math.log(prob) + math.log(emission_prob)

        return log_prob


    def brute_force_decode(self, sentence):
        """
        Find the best tag sequence using brute force approach

        Ալգորիթմ:
        1. Գեներացնել բոլոր հնարավոր tag sequences-ները
        2. Հաշվել յուրաքանչյուրի հավանականությունը
        3. Վերադարձնել ամենամեծ հավանականությունը ունեցող sequence-ը

        Time Complexity: O(T^N) որտեղ T = թույլատրված tags-երի քանակը, N = բառերի քանակը

        Args:
            sentence: List of words, e.g., ["The", "dog", "barks"]

        Returns:
            List of tags, e.g., ["DET", "NOUN", "VERB"]
        """

        if not self.trained:
            raise ValueError("Model must be trained before decoding!")

        if isinstance(sentence, str):
            sentence = sentence.split()

        n = len(sentence)
        all_tags_list = list(self.all_tags)

        print(f"Decoding sentence: {' '.join(sentence)}")
        print(f"Sentence length: {n} words")
        print(f"Number of possible tag sequences: {len(all_tags_list)}^{n} = {len(all_tags_list)**n:,}")

        # Գեներացնել բոլոր հնարավոր tag sequences-ները
        # product() ստեղծում է Cartesian product
        all_possible_sequences = product(all_tags_list, repeat=n)

        best_sequence = None
        best_prob = float('-inf')  # Log space-ում սկսում ենք -∞-ից

        sequences_checked = 0

        # Ստուգել յուրաքանչյուր հնարավոր sequence
        for tag_sequence in all_possible_sequences:
            sequences_checked += 1

            # Հաշվել այս sequence-ի հավանականությունը
            prob = self.calculate_sequence_probability(sentence, tag_sequence)

            # Եթե սա ավելի լավ է, թարմացնել
            if prob > best_prob:
                best_prob = prob
                best_sequence = tag_sequence

        print(f"Checked {sequences_checked:,} sequences")
        print(f"Best sequence probability (log): {best_prob:.4f}")
        print()

        return list(best_sequence)


    def tag_sentence(self, sentence):
        """
        Tag a sentence and return (word, tag) pairs

        Args:
            sentence: List of words or string

        Returns:
            List of (word, tag) tuples
        """

        if isinstance(sentence, str):
            sentence = sentence.split()

        tags = self.brute_force_decode(sentence)
        return list(zip(sentence, tags))


# ============================================================================
# EXAMPLE USAGE AND TESTING
# ============================================================================

def main():
    """
    Demonstrate the HMM POS Tagger with Brute Force Decoding
    """

    print("=" * 80)
    print("HMM POS TAGGER - BRUTE FORCE DECODING")
    print("=" * 80)
    print()

    # TRAINING DATA
    training_data = [
        [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
        [("The", "DET"), ("cat", "NOUN"), ("meows", "VERB")],
        [("A", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
        [("The", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
        [("A", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")],
        [("The", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("barks", "VERB")],
        [("A", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("meows", "VERB")],
        [("The", "DET"), ("dog", "NOUN"), ("sleeps", "VERB")],
        [("The", "DET"), ("cat", "NOUN"), ("runs", "VERB")],
        [("A", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("sleeps", "VERB")],
        [("The", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("runs", "VERB")],
        [("Dogs", "NOUN"), ("bark", "VERB")],
        [("Cats", "NOUN"), ("meow", "VERB")],
        [("The", "DET"), ("dog", "NOUN"), ("in", "ADP"), ("the", "DET"), ("house", "NOUN"), ("barks", "VERB")],
        [("A", "DET"), ("cat", "NOUN"), ("on", "ADP"), ("the", "DET"), ("mat", "NOUN"), ("sleeps", "VERB")],
    ]

    # CREATE AND TRAIN TAGGER
    tagger = HMMPOSTagger(smoothing=1.0)
    tagger.train(training_data)

    # TEST SENTENCES
    print("=" * 80)
    print("TESTING ON NEW SENTENCES")
    print("=" * 80)
    print()

    test_sentences = [
        ["The", "dog", "barks"],
        ["A", "cat", "sleeps"],
        ["The", "big", "dog", "runs"],
        ["Dogs", "bark"],
    ]

    for sentence in test_sentences:
        tagged = tagger.tag_sentence(sentence)

        print("Result:")
        for word, tag in tagged:
            print(f"  {word:10} -> {tag}")
        print()
        print("-" * 80)
        print()

    # COMPLEXITY ANALYSIS
    print("=" * 80)
    print("COMPLEXITY ANALYSIS")
    print("=" * 80)
    print()
    print("Brute Force Decoding:")
    print("  Time Complexity: O(T^N × N)")
    print("    where T = number of tags")
    print("          N = sentence length")
    print()
    print("  For our example:")
    print(f"    T = {len(tagger.all_tags)} tags")
    print(f"    N = 3 words -> {len(tagger.all_tags)}^3 = {len(tagger.all_tags)**3} sequences to check")
    print(f"    N = 4 words -> {len(tagger.all_tags)}^4 = {len(tagger.all_tags)**4} sequences to check")
    print()
    print("  This grows EXPONENTIALLY! 🚀")
    print()
    print("Viterbi Algorithm (Dynamic Programming):")
    print("  Time Complexity: O(T^2 × N)")
    print("  Much more efficient for longer sentences!")
    print()


if __name__ == "__main__":
    main()

HMM POS TAGGER - BRUTE FORCE DECODING

Training HMM POS Tagger...
Training on 15 sentences
Training complete!
Learned 5 POS tags: ['ADJ', 'ADP', 'DET', 'NOUN', 'VERB']
Vocabulary size: 19 words

TESTING ON NEW SENTENCES

Decoding sentence: The dog barks
Sentence length: 3 words
Number of possible tag sequences: 5^3 = 125
Checked 125 sequences
Best sequence probability (log): -6.1592

Result:
  The        -> DET
  dog        -> NOUN
  barks      -> VERB

--------------------------------------------------------------------------------

Decoding sentence: A cat sleeps
Sentence length: 3 words
Number of possible tag sequences: 5^3 = 125
Checked 125 sequences
Best sequence probability (log): -6.4751

Result:
  A          -> DET
  cat        -> NOUN
  sleeps     -> VERB

--------------------------------------------------------------------------------

Decoding sentence: The big dog runs
Sentence length: 4 words
Number of possible tag sequences: 5^4 = 625
Checked 625 sequences
Best sequence p